In [ ]:
import pandas as pd

# Load restaurant dataset
restaurants = pd.read_csv(
    "data/raw/Restaurant names and Metadata.csv"
)

# Load customer reviews dataset
reviews = pd.read_csv(
    "data/raw/Restaurant reviews_public.csv"
)

print("Datasets loaded successfully!")

In [ ]:
restaurants.head()

In [ ]:
reviews.head()

In [ ]:
restaurants.shape()


In [ ]:
restaurants.shape

In [ ]:
reviews.shape

In [ ]:
restaurants.info()


In [ ]:
reviews.info

In [ ]:
restaurants.isnull().sum()

In [ ]:
reviews.isnull().sum()

In [ ]:
restaurants.duplicated().sum()

In [ ]:
reviews.duplicated().sum()

In [ ]:
print("RESTAURANT DATA TYPES")
print(restaurants.dtypes)

print("\nREVIEW DATA TYPES")
print(reviews.dtypes)

In [ ]:
reviews["Rating"].value_counts(dropna=False)

In [ ]:
reviews["Review"].astype("string").str.strip().eq("").sum()

In [ ]:
restaurant_names = set(
    restaurants["Name"].str.strip().str.casefold()
)

review_names = set(
    reviews["Restaurant"].str.strip().str.casefold()
)

print("Restaurants in metadata:", len(restaurant_names))
print("Restaurants in reviews:", len(review_names))

print(
    "Reviews restaurants missing from metadata:",
    review_names - restaurant_names
)

In [ ]:
rating_numeric = pd.to_numeric(
    reviews["Rating"],
    errors="coerce"
)

print("Original ratings:", reviews["Rating"].nunique())

print("Ratings that cannot be converted:")
print(
    reviews.loc[
        rating_numeric.isna() & reviews["Rating"].notna(),
        "Rating"
    ].unique()
)

In [ ]:
restaurants_without_reviews = (
    restaurant_names - review_names
)

print("Restaurants without reviews:")
print(restaurants_without_reviews)

In [ ]:
print("Minimum rating:", rating_numeric.min())
print("Maximum rating:", rating_numeric.max())

print("Missing or invalid ratings:", rating_numeric.isna().sum())

In [ ]:
restaurants["Cost"].unique()

In [ ]:
rating_numeric.value_counts().sort_index()

In [ ]:
print("Restaurant missing values:")
print(restaurants.isnull().sum())

print("\nReview missing values:")
print(reviews.isnull().sum())

In [ ]:
restaurants["Cost"].unique()

In [ ]:
rating_numeric.value_counts().sort_index()

In [ ]:
print("Restaurant missing values:")
print(restaurants.isnull().sum())

print("\nReview missing values:")
print(reviews.isnull().sum())

In [ ]:
# Create copies of original datasets
restaurants_clean = restaurants.copy()
reviews_clean = reviews.copy()

print("Clean copies created successfully!")

In [ ]:
# Remove commas from cost and convert to numeric
restaurants_clean["Cost"] = pd.to_numeric(
    restaurants_clean["Cost"].str.replace(",", "", regex=False),
    errors="coerce"
)

print(restaurants_clean["Cost"].dtype)
print(restaurants_clean["Cost"].head())

In [ ]:
# Convert ratings into numeric values
reviews_clean["Rating"] = pd.to_numeric(
    reviews_clean["Rating"],
    errors="coerce"
)

print(reviews_clean["Rating"].dtype)
print(reviews_clean["Rating"].isnull().sum())

In [ ]:
# Count duplicates before removal
print("Duplicates before cleaning:",
      reviews_clean.duplicated().sum())

# Remove exact duplicate rows
reviews_clean = reviews_clean.drop_duplicates()

# Check again
print("Duplicates after cleaning:",
      reviews_clean.duplicated().sum())

In [ ]:
# Keep only rows containing actual review text
sentiment_data = reviews_clean.dropna(
    subset=["Review"]
).copy()

print("Reviews available for sentiment analysis:",
      len(sentiment_data))

print("Missing review text:",
      sentiment_data["Review"].isnull().sum())

In [ ]:
from pathlib import Path

# Make sure the processed folder exists
Path("data/processed").mkdir(
    parents=True,
    exist_ok=True
)

# Save cleaned datasets
restaurants_clean.to_csv(
    "data/processed/restaurants_clean.csv",
    index=False
)

reviews_clean.to_csv(
    "data/processed/reviews_clean.csv",
    index=False
)

sentiment_data.to_csv(
    "data/processed/sentiment_data.csv",
    index=False
)

print("All cleaned datasets saved successfully!")

In [ ]:
print("Restaurant dataset:", restaurants_clean.shape)
print("Review dataset:", reviews_clean.shape)
print("Sentiment dataset:", sentiment_data.shape)

print("\nRestaurant cost type:")
print(restaurants_clean["Cost"].dtype)

print("\nRating type:")
print(reviews_clean["Rating"].dtype)

print("\nRemaining duplicate reviews:")
print(reviews_clean.duplicated().sum())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set a consistent chart style
sns.set_theme(style="whitegrid")

print("Visualization libraries imported successfully!")

In [ ]:
print("Restaurant Cost Statistics")
print(restaurants_clean["Cost"].describe())

In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(
    data=restaurants_clean,
    x="Cost",
    bins=15,
    kde=True
)

plt.title("Distribution of Restaurant Costs")
plt.xlabel("Restaurant Cost")
plt.ylabel("Number of Restaurants")

plt.tight_layout()
plt.show()

In [ ]:
# Split comma-separated cuisines into individual values
cuisine_series = (
    restaurants_clean["Cuisines"]
    .dropna()
    .str.split(",")
    .explode()
    .str.strip()
)

# Count each cuisine
cuisine_counts = cuisine_series.value_counts()

print("Top 10 cuisines:")
print(cuisine_counts.head(10))

In [ ]:
plt.figure(figsize=(11, 6))

sns.barplot(
    x=cuisine_counts.head(10).values,
    y=cuisine_counts.head(10).index
)

plt.title("Top 10 Most Common Cuisines")
plt.xlabel("Number of Restaurants")
plt.ylabel("Cuisine")

plt.tight_layout()
plt.show()

In [ ]:
rating_counts = reviews_clean["Rating"].value_counts().sort_index()

print(rating_counts)

In [ ]:
plt.figure(figsize=(9, 5))

sns.countplot(
    data=reviews_clean,
    x="Rating",
    order=sorted(reviews_clean["Rating"].dropna().unique())
)

plt.title("Distribution of Customer Ratings")
plt.xlabel("Rating")
plt.ylabel("Number of Reviews")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

print("EDA setup completed!")

In [ ]:
restaurants_clean["Cost"].describe()

In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(
    data=restaurants_clean,
    x="Cost",
    bins=15,
    kde=True
)

plt.title("Distribution of Restaurant Costs")
plt.xlabel("Restaurant Cost")
plt.ylabel("Number of Restaurants")

plt.tight_layout()
plt.show()

In [ ]:
# Convert ratings into numeric values
reviews_clean["Rating"] = pd.to_numeric(
    reviews_clean["Rating"],
    errors="coerce"
)

# Calculate average rating for each restaurant
restaurant_ratings = (
    reviews_clean
    .groupby("Restaurant")["Rating"]
    .mean()
    .sort_values(ascending=False)
)

print("Top 10 highest-rated restaurants:")
print(restaurant_ratings.head(10))

print("\nOverall average rating:")
print(reviews_clean["Rating"].mean())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(9, 5))

sns.histplot(
    data=reviews_clean,
    x="Rating",
    bins=10,
    kde=True
)

plt.title("Distribution of Customer Ratings")
plt.xlabel("Customer Rating")
plt.ylabel("Number of Reviews")

plt.tight_layout()
plt.show()

In [ ]:
# Create a copy of the cleaned restaurant dataset
restaurant_features = restaurants_clean.copy()

# Count the number of cuisines offered by each restaurant
restaurant_features["Cuisine_Count"] = (
    restaurant_features["Cuisines"]
    .str.split(",")
    .str.len()
)

# Calculate average rating and total reviews for each restaurant
rating_summary = (
    reviews_clean
    .groupby("Restaurant")["Rating"]
    .agg(
        Average_Rating="mean",
        Total_Reviews="count"
    )
    .reset_index()
)

# Merge restaurant information with rating information
restaurant_features = restaurant_features.merge(
    rating_summary,
    left_on="Name",
    right_on="Restaurant",
    how="left"
)

# Remove the extra Restaurant column
restaurant_features.drop(columns=["Restaurant"], inplace=True)

# Display the new features
restaurant_features.head()

In [ ]:
print("New dataset shape:", restaurant_features.shape)

print("\nNew columns:")
print(restaurant_features.columns.tolist())

print("\nMissing values:")
print(restaurant_features.isnull().sum())

In [ ]:
restaurant_features.to_csv(
    "data/processed/restaurant_features.csv",
    index=False
)

print("Feature-engineered dataset saved successfully!")

In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns

# Select restaurants that have valid average ratings
top_restaurants = (
    restaurant_features
    .dropna(subset=["Average_Rating"])
    .nlargest(10, "Average_Rating")
    .sort_values("Average_Rating")
)

# Create the chart
plt.figure(figsize=(10, 6))

ax = sns.barplot(
    data=top_restaurants,
    x="Average_Rating",
    y="Name"
)

# Add rating values to the bars
ax.bar_label(ax.containers[0], fmt="%.2f", padding=5)

plt.title("Top 10 Restaurants by Average Customer Rating")
plt.xlabel("Average Customer Rating")
plt.ylabel("Restaurant Name")
plt.xlim(0, 5)

plt.tight_layout()
plt.show()


In [ ]:
print("Python is working!")

print("restaurant_features exists:",
      "restaurant_features" in globals())

In [ ]:
# Create restaurant features from cleaned data
restaurant_features = restaurants_clean.copy()

# Count cuisines
restaurant_features["Cuisine_Count"] = (
    restaurant_features["Cuisines"]
    .str.split(",")
    .str.len()
)

# Calculate average rating and total reviews
rating_summary = (
    reviews_clean
    .groupby("Restaurant")["Rating"]
    .agg(
        Average_Rating="mean",
        Total_Reviews="count"
    )
    .reset_index()
)

# Merge both datasets
restaurant_features = restaurant_features.merge(
    rating_summary,
    left_on="Name",
    right_on="Restaurant",
    how="left"
)

# Remove extra column
restaurant_features = restaurant_features.drop(
    columns=["Restaurant"]
)

print("Restaurant features recreated successfully!")
print("Dataset shape:", restaurant_features.shape)

restaurant_features.head()

In [ ]:
print("restaurant_features exists:",
      "restaurant_features" in globals())

In [ ]:
# Start with restaurant metadata
restaurant_features = restaurants_clean.copy()

# Count the number of cuisines offered by each restaurant
restaurant_features["Cuisine_Count"] = (
    restaurant_features["Cuisines"]
    .str.split(",")
    .str.len()
)

# Calculate average rating and total reviews per restaurant
rating_summary = (
    reviews_clean
    .groupby("Restaurant")["Rating"]
    .agg(
        Average_Rating="mean",
        Total_Reviews="count"
    )
    .reset_index()
)

# Merge restaurant information with rating information
restaurant_features = restaurant_features.merge(
    rating_summary,
    left_on="Name",
    right_on="Restaurant",
    how="left"
)

# Remove the extra Restaurant column
restaurant_features = restaurant_features.drop(
    columns=["Restaurant"]
)

# Save the recreated features
restaurant_features.to_csv(
    "data/processed/restaurant_features.csv",
    index=False
)

print("Restaurant features recreated successfully!")
print("Dataset shape:", restaurant_features.shape)

restaurant_features.head()

In [ ]:
print("restaurant_features exists:",
      "restaurant_features" in globals())

In [ ]:
import pandas as pd

restaurants_clean = pd.read_csv(
    "data/processed/restaurants_clean.csv"
)

reviews_clean = pd.read_csv(
    "data/processed/reviews_clean.csv"
)

reviews_clean["Rating"] = pd.to_numeric(
    reviews_clean["Rating"],
    errors="coerce"
)

print("Datasets loaded successfully!")
print("Restaurants:", restaurants_clean.shape)
print("Reviews:", reviews_clean.shape)

In [ ]:
# Select the top 10 restaurants by average rating
top_restaurants = (
    restaurant_features
    .dropna(subset=["Average_Rating"])
    .nlargest(10, "Average_Rating")
    .sort_values("Average_Rating")
)

# Create the bar chart
plt.figure(figsize=(10, 6))

ax = sns.barplot(
    data=top_restaurants,
    x="Average_Rating",
    y="Name"
)

ax.bar_label(
    ax.containers[0],
    fmt="%.2f",
    padding=5
)

plt.title("Top 10 Restaurants by Average Customer Rating")
plt.xlabel("Average Customer Rating")
plt.ylabel("Restaurant Name")
plt.xlim(0, 5)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np

print("Libraries imported successfully!")


In [ ]:
# Load restaurant metadata
restaurants = pd.read_csv(
    "../data/raw/Restaurant names and Metadata.csv"
)

# Load customer reviews
reviews = pd.read_csv(
    "../data/raw/Restaurant reviews_public.csv"
)

print("Both datasets loaded successfully!")

In [ ]:
print("RESTAURANT DATASET")
print("Shape:", restaurants.shape)

print("\nREVIEW DATASET")
print("Shape:", reviews.shape)

In [ ]:
print("RESTAURANT COLUMNS:")
print(restaurants.columns.tolist())

print("\nREVIEW COLUMNS:")
print(reviews.columns.tolist())

In [ ]:
display(restaurants.head())
display(reviews.head())

In [ ]:
print("RESTAURANT MISSING VALUES:")
print(restaurants.isnull().sum())

print("\nREVIEW MISSING VALUES:")
print(reviews.isnull().sum())

In [ ]:
from pathlib import Path
import pandas as pd

# Check possible project locations
possible_roots = [Path.cwd(), Path.cwd().parent]

project_root = None

for folder in possible_roots:
    restaurant_file = folder / "data/raw/Restaurant names and Metadata.csv"
    review_file = folder / "data/raw/Restaurant reviews_public.csv"

    if restaurant_file.exists() and review_file.exists():
        project_root = folder
        break

if project_root is None:
    raise FileNotFoundError(
        f"Datasets not found. Current directory: {Path.cwd()}"
    )

# Load both datasets
restaurants = pd.read_csv(
    project_root / "data/raw/Restaurant names and Metadata.csv"
)

reviews = pd.read_csv(
    project_root / "data/raw/Restaurant reviews_public.csv"
)

print("Both datasets loaded successfully!")
print("Project location:", project_root.resolve())
print("Restaurant shape:", restaurants.shape)
print("Reviews shape:", reviews.shape)

In [ ]:
# Check restaurant dataset information
print("RESTAURANT DATASET INFO")
restaurants.info()

# Check review dataset information
print("\nREVIEW DATASET INFO")
reviews.info()

In [ ]:
# Check duplicate records
print("Restaurant duplicates:",
      restaurants.duplicated().sum())

print("Review duplicates:",
      reviews.duplicated().sum())

In [ ]:
# Inspect different rating values
print("Unique review ratings:")
print(reviews["Rating"].value_counts(dropna=False))

In [ ]:
# Find reviews with non-numeric rating values

like_reviews = reviews[
    reviews["Rating"].astype(str).str.strip().str.lower() == "like"
]

display(like_reviews[["Restaurant", "Review", "Rating"]])

In [ ]:
# Convert rating values to numeric
rating_numeric = pd.to_numeric(
    reviews["Rating"],
    errors="coerce"
)

print("Numeric ratings:", rating_numeric.notna().sum())
print("Non-numeric or missing ratings:", rating_numeric.isna().sum())

print("\nRating statistics:")
print(rating_numeric.describe())

In [ ]:
# Create a copy of the original reviews dataset
reviews_analysis = reviews.copy()

# Convert Rating into numeric format
reviews_analysis["Rating_numeric"] = pd.to_numeric(
    reviews_analysis["Rating"],
    errors="coerce"
)

# Check the result
print("Original dataset shape:", reviews.shape)
print("Analysis dataset shape:", reviews_analysis.shape)

print("\nMissing numeric ratings:")
print(reviews_analysis["Rating_numeric"].isna().sum())

In [ ]:
# Inspect some customer reviews
display(
    reviews_analysis[
        ["Restaurant", "Review", "Rating", "Rating_numeric"]
    ].head(10)
)

In [ ]:
# Count missing review text
print(
    "Missing review texts:",
    reviews_analysis["Review"].isna().sum()
)

# Count empty or whitespace-only reviews
empty_reviews = (
    reviews_analysis["Review"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

print("Empty review texts:", empty_reviews)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Find project directory
project_root = Path.cwd()

if not (project_root / "data/raw").exists():
    project_root = project_root.parent

# Load restaurant dataset
restaurants = pd.read_csv(
    project_root / "data/raw/Restaurant names and Metadata.csv"
)

# Load customer reviews
reviews = pd.read_csv(
    project_root / "data/raw/Restaurant reviews_public.csv"
)

print("Datasets loaded successfully!")
print("Restaurants:", restaurants.shape)
print("Reviews:", reviews.shape)

In [ ]:
# Create analysis copy
reviews_analysis = reviews.copy()

# Convert ratings into numeric values
reviews_analysis["Rating_numeric"] = pd.to_numeric(
    reviews_analysis["Rating"],
    errors="coerce"
)

print("Analysis dataset created successfully!")
print("Shape:", reviews_analysis.shape)

In [ ]:
print("reviews exists:", "reviews" in globals())
print("reviews_analysis exists:", "reviews_analysis" in globals())

In [ ]:
# Count missing review text
print(
    "Missing review texts:",
    reviews_analysis["Review"].isna().sum()
)

# Count empty or whitespace-only reviews
empty_reviews = (
    reviews_analysis["Review"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

print("Empty review texts:", empty_reviews)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Find the project folder
possible_roots = [Path.cwd(), Path.cwd().parent]
project_root = None

for folder in possible_roots:
    if (folder / "data/raw/Restaurant names and Metadata.csv").exists():
        project_root = folder
        break

if project_root is None:
    raise FileNotFoundError("Could not find the project data folder.")

# Load datasets
restaurants = pd.read_csv(
    project_root / "data/raw/Restaurant names and Metadata.csv"
)
reviews = pd.read_csv(
    project_root / "data/raw/Restaurant reviews_public.csv"
)

# Create a separate copy for review analysis
reviews_analysis = reviews.copy()

# Convert ratings to numeric values
reviews_analysis["Rating_numeric"] = pd.to_numeric(
    reviews_analysis["Rating"],
    errors="coerce"
)

# Display results
print("Restaurants shape:", restaurants.shape)
print("Reviews shape:", reviews.shape)
print("Missing review texts:", reviews_analysis["Review"].isna().sum())
print(
    "Empty review texts:",
    reviews_analysis["Review"].fillna("").str.strip().eq("").sum()
)

display(
    reviews_analysis[
        ["Restaurant", "Review", "Rating", "Rating_numeric"]
    ].head(10)
)

In [ ]:
# Create a cleaned copy
reviews_clean = reviews_analysis.copy()

# Remove rows where review text is missing
reviews_clean = reviews_clean.dropna(subset=["Review"])

# Remove rows where review text is empty or contains only spaces
reviews_clean = reviews_clean[
    reviews_clean["Review"].str.strip() != ""
].copy()

# Reset row numbering
reviews_clean = reviews_clean.reset_index(drop=True)


# Check results
print("Original reviews:", len(reviews_analysis))
print("Clean reviews:", len(reviews_clean))
print("Removed reviews:", len(reviews_analysis) - len(reviews_clean))
print("Remaining missing reviews:", reviews_clean["Review"].isna().sum())

In [ ]:
# Check a few original reviews
for review in reviews_clean["Review"].head(5):
    print(review)
    print("-" * 80)

In [ ]:
import re

def clean_review(text):

    # Convert text to lowercase
    text = text.lower()

    # Remove HTML tags
    text = re.sub(r"<.*?>", " ", text)

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Remove punctuation and special characters
    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


# Apply cleaning to all reviews
reviews_clean["Clean_Review"] = reviews_clean["Review"].apply(clean_review)

# Compare original and cleaned reviews
display(
    reviews_clean[["Review", "Clean_Review"]].head(5)
)

In [ ]:
# Check for missing cleaned reviews
print(
    "Missing cleaned reviews:",
    reviews_clean["Clean_Review"].isna().sum()
)

# Check for empty cleaned reviews
print(
    "Empty cleaned reviews:",
    reviews_clean["Clean_Review"].str.strip().eq("").sum()
)

# Check dataset dimensions
print("Final review dataset shape:", reviews_clean.shape)

In [ ]:
# Count empty cleaned reviews before removal
empty_count = reviews_clean["Clean_Review"].str.strip().eq("").sum()

# Remove empty cleaned reviews
reviews_clean = reviews_clean[
    reviews_clean["Clean_Review"].str.strip() != ""
].copy()

# Reset index
reviews_clean = reviews_clean.reset_index(drop=True)

# Verify
print("Empty reviews removed:", empty_count)
print("Remaining reviews:", len(reviews_clean))
print(
    "Remaining empty reviews:",
    reviews_clean["Clean_Review"].str.strip().eq("").sum()
)

In [ ]:
# Count each rating
rating_distribution = (
    reviews_clean["Rating_numeric"]
    .value_counts()
    .sort_index()
)

print("Rating distribution:")
display(rating_distribution)

# Calculate average rating
print(
    "Average rating:",
    round(reviews_clean["Rating_numeric"].mean(), 2)
)

# Count missing numeric ratings
print(
    "Missing numeric ratings:",
    reviews_clean["Rating_numeric"].isna().sum()
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 5))

sns.countplot(
    data=reviews_clean,
    x="Rating_numeric",
    order=sorted(reviews_clean["Rating_numeric"].dropna().unique()),
    color="coral"
)

plt.title("Distribution of Customer Ratings")
plt.xlabel("Customer Rating")
plt.ylabel("Number of Reviews")
plt.tight_layout()

plt.savefig(
    project_root / "outputs/charts/rating_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Create the sentiment analyzer
analyzer = SentimentIntensityAnalyzer()

print("VADER initialized successfully!")

In [ ]:
# Sample customer reviews
sample_reviews = [
    "The food was absolutely amazing!",
    "The food was okay.",
    "The food was terrible and disappointing."
]

# Analyze each review
for review in sample_reviews:

    scores = analyzer.polarity_scores(review)

    print("Review:", review)
    print("Sentiment scores:", scores)
    print("-" * 60)

In [ ]:
# Calculate sentiment scores for every cleaned review

sentiment_scores = reviews_clean["Clean_Review"].apply(
    analyzer.polarity_scores
)

# Convert scores into separate columns
reviews_clean["Positive_Score"] = sentiment_scores.apply(
    lambda x: x["pos"]
)

reviews_clean["Negative_Score"] = sentiment_scores.apply(
    lambda x: x["neg"]
)

reviews_clean["Neutral_Score"] = sentiment_scores.apply(
    lambda x: x["neu"]
)

reviews_clean["Compound_Score"] = sentiment_scores.apply(
    lambda x: x["compound"]
)

print("Sentiment scores calculated successfully!")

display(
    reviews_clean[
        ["Review", "Compound_Score"]
    ].head(10)
)

In [ ]:
# Classify reviews using VADER compound scores

reviews_clean["Sentiment"] = np.where(
    reviews_clean["Compound_Score"] >= 0.05,
    "Positive",
    np.where(
        reviews_clean["Compound_Score"] <= -0.05,
        "Negative",
        "Neutral"
    )
)

# Display sample results
display(
    reviews_clean[
        ["Review", "Compound_Score", "Sentiment"]
    ].head(10)
)

# Count each sentiment category
sentiment_counts = reviews_clean["Sentiment"].value_counts()

print("\nSentiment Distribution:")
display(sentiment_counts)

In [ ]:
# Count positive, negative, and neutral reviews
sentiment_counts = reviews_clean["Sentiment"].value_counts()

print("Sentiment Distribution:")
display(sentiment_counts)

# Calculate percentage distribution
sentiment_percentage = (
    reviews_clean["Sentiment"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nSentiment Percentage:")
display(sentiment_percentage)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Calculate sentiment counts
sentiment_counts = reviews_clean["Sentiment"].value_counts()

# Create output folder if it doesn't exist
(project_root / "outputs/charts").mkdir(
    parents=True,
    exist_ok=True
)

# Create chart
plt.figure(figsize=(9, 5))

sns.barplot(
    x=sentiment_counts.index,
    y=sentiment_counts.values
)

plt.title("Customer Review Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Number of Reviews")
plt.tight_layout()

# Save chart
plt.savefig(
    project_root / "outputs/charts/sentiment_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Inspect restaurant metadata
print("Restaurant dataset shape:", restaurants.shape)

print("\nColumn information:")
print(restaurants.info())

print("\nFirst five restaurants:")
display(restaurants.head())

print("\nMissing values:")
display(restaurants.isnull().sum())

In [ ]:
# Check data types
print("Column data types:")
print(restaurants.dtypes)

# Inspect cost values
print("\nCost examples:")
display(restaurants[["Name", "Cost"]].head(10))

# Inspect cuisine values
print("\nCuisine examples:")
display(restaurants[["Name", "Cuisines"]].head(5))

# Inspect collection values
print("\nCollection examples:")
display(restaurants[["Name", "Collections"]].head(5))


In [ ]:
# Create a copy of the restaurant dataset
restaurant_features = restaurants.copy()

# Count the number of cuisines offered
restaurant_features["Cuisine_Count"] = (
    restaurant_features["Cuisines"]
    .str.split(",")
    .str.len()
)

# Count restaurant collection tags
restaurant_features["Collection_Count"] = (
    restaurant_features["Collections"]
    .fillna("")
    .apply(
        lambda x: len([
            item for item in x.split(",")
            if item.strip()
        ])
    )
)

# Display engineered features
display(
    restaurant_features[
        ["Name", "Cost", "Cuisine_Count", "Collection_Count"]
    ].head(10)
)

In [ ]:
print("Feature dataset shape:", restaurant_features.shape)

print("\nMissing values:")
display(
    restaurant_features[
        ["Cost", "Cuisine_Count", "Collection_Count"]
    ].isnull().sum()
)

print("\nFeature statistics:")
display(
    restaurant_features[
        ["Cost", "Cuisine_Count", "Collection_Count"]
    ].describe()
)

In [ ]:
print("Data types:")
print(restaurant_features[["Cost", "Cuisine_Count", "Collection_Count"]].dtypes)

print("\nCost statistics:")
display(restaurant_features["Cost"].describe())

print("\nCost values:")
display(restaurant_features[["Name", "Cost"]].head(10))


In [ ]:
# Convert Cost from string to numeric
restaurant_features["Cost"] = pd.to_numeric(
    restaurant_features["Cost"]
    .astype(str)
    .str.replace(",", "", regex=False),
    errors="coerce"
)

# Check the conversion
print("Updated data types:")
print(restaurant_features[["Cost", "Cuisine_Count", "Collection_Count"]].dtypes)

print("\nMissing values:")
print(
    restaurant_features[
        ["Cost", "Cuisine_Count", "Collection_Count"]
    ].isnull().sum()
)

print("\nUpdated feature statistics:")
display(
    restaurant_features[
        ["Cost", "Cuisine_Count", "Collection_Count"]
    ].describe()
)

In [ ]:
features = ["Cost", "Cuisine_Count", "Collection_Count"]

print("Data types:")
print(restaurant_features[features].dtypes)

print("\nMissing values:")
print(restaurant_features[features].isnull().sum())

print("\nFeature statistics:")
display(restaurant_features[features].describe())


In [ ]:
from sklearn.preprocessing import StandardScaler

# Select numerical features
X = restaurant_features[features].copy()

# Create scaler
scaler = StandardScaler()

# Transform features to a common scale
X_scaled = scaler.fit_transform(X)

print("Original feature shape:", X.shape)
print("Scaled feature shape:", X_scaled.shape)

# Display scaled data
display(pd.DataFrame(X_scaled, columns=features).head())

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# Store WCSS values
wcss = []

# Test different numbers of clusters
for k in range(1, 11):
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    kmeans.fit(X_scaled)

    wcss.append(kmeans.inertia_)

# Plot the Elbow Curve
plt.figure(figsize=(9, 5))

plt.plot(
    range(1, 11),
    wcss,
    marker="o"
)

plt.title("Elbow Method for Optimal K")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("WCSS (Inertia)")
plt.xticks(range(1, 11))
plt.grid(True)
plt.tight_layout()

plt.savefig(
    project_root / "outputs/charts/elbow_method.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
for k, value in enumerate(wcss, start=1):
    print(f"K = {k}: WCSS = {value:.2f}")

In [ ]:
from sklearn.metrics import silhouette_score

silhouette_results = []

for k in [3, 4, 5]:
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = model.fit_predict(X_scaled)

    score = silhouette_score(X_scaled, labels)

    silhouette_results.append({
        "K": k,
        "WCSS": round(model.inertia_, 2),
        "Silhouette Score": round(score, 4)
    })

silhouette_df = pd.DataFrame(silhouette_results)

display(silhouette_df)

In [ ]:
# Train final K-Means model
final_kmeans = KMeans(
    n_clusters=5,
    random_state=42,
    n_init=10
)

# Assign cluster labels
restaurant_features["Cluster"] = final_kmeans.fit_predict(X_scaled)

# Check cluster distribution
print("Restaurant count in each cluster:")
display(
    restaurant_features["Cluster"]
    .value_counts()
    .sort_index()
    .rename("Restaurant_Count")
    .to_frame()
)

print("Final Silhouette Score:",
      round(silhouette_score(X_scaled, restaurant_features["Cluster"]), 4))

In [ ]:
cluster_summary = (
    restaurant_features
    .groupby("Cluster")
    .agg(
        Restaurant_Count=("Name", "count"),
        Average_Cost=("Cost", "mean"),
        Average_Cuisines=("Cuisine_Count", "mean"),
        Average_Collections=("Collection_Count", "mean")
    )
    .round(2)
)

display(cluster_summary)

In [ ]:
print("1. Number of restaurants in each cluster:")
display(
    restaurant_features["Cluster"]
    .value_counts()
    .sort_index()
)

print("\n2. Restaurant cluster summary:")
display(cluster_summary)

print("\n3. Sample restaurants with cluster assignments:")
display(
    restaurant_features[
        ["Name", "Cost", "Cuisine_Count",
         "Collection_Count", "Cluster"]
    ].sort_values("Cluster").head(15)
)

In [ ]:
print("CLUSTER DISTRIBUTION")
print(restaurant_features["Cluster"].value_counts().sort_index())

print("\nCLUSTER SUMMARY")
display(cluster_summary)

In [ ]:
print("CLUSTER DISTRIBUTION")
display(
    restaurant_features["Cluster"]
    .value_counts()
    .sort_index()
    .to_frame("Restaurant_Count")
)

print("\nCLUSTER SUMMARY")
display(cluster_summary)

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Reduce 3 features to 2 dimensions
pca = PCA(n_components=2)

X_pca = pca.fit_transform(X_scaled)

# Create a DataFrame for visualization
pca_df = pd.DataFrame({
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1],
    "Cluster": restaurant_features["Cluster"]
})

# Plot clusters
plt.figure(figsize=(10, 6))

for cluster in sorted(pca_df["Cluster"].unique()):
    cluster_data = pca_df[pca_df["Cluster"] == cluster]

    plt.scatter(
        cluster_data["PC1"],
        cluster_data["PC2"],
        label=f"Cluster {cluster}",
        s=70,
        alpha=0.75
    )

plt.title("Restaurant Segmentation using K-Means and PCA")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()

plt.savefig(
    project_root / "outputs/charts/restaurant_clusters_pca.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Variance explained by PCA:",
      round(pca.explained_variance_ratio_.sum() * 100, 2), "%")

In [ ]:
processed_dir = project_root / "data/processed"
processed_dir.mkdir(parents=True, exist_ok=True)

restaurant_features.to_csv(
    processed_dir / "restaurants_clustered.csv",
    index=False
)

print("Clustered dataset saved successfully!")

In [ ]:
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

# Calculate evaluation metrics
silhouette = silhouette_score(
    X_scaled,
    restaurant_features["Cluster"]
)

davies_bouldin = davies_bouldin_score(
    X_scaled,
    restaurant_features["Cluster"]
)

calinski_harabasz = calinski_harabasz_score(
    X_scaled,
    restaurant_features["Cluster"]
)

print("K-MEANS MODEL EVALUATION")
print("------------------------")
print(f"Silhouette Score: {silhouette:.4f}")
print(f"Davies-Bouldin Index: {davies_bouldin:.4f}")
print(f"Calinski-Harabasz Score: {calinski_harabasz:.2f}")

In [ ]:
import joblib

models_dir = project_root / "models"
models_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(
    final_kmeans,
    models_dir / "kmeans_model.pkl"
)

joblib.dump(
    scaler,
    models_dir / "scaler.pkl"
)

print("K-Means model saved!")
print("Scaler saved!")

In [ ]:
print("K-MEANS MODEL EVALUATION")
print("------------------------")

print(
    "Silhouette Score:",
    round(silhouette, 4)
)

print(
    "Davies-Bouldin Index:",
    round(davies_bouldin, 4)
)

print(
    "Calinski-Harabasz Score:",
    round(calinski_harabasz, 2)
)

In [ ]:
print("K-Means model exists:",
      (models_dir / "kmeans_model.pkl").exists())

print("Scaler exists:",
      (models_dir / "scaler.pkl").exists())

print("Clustered dataset exists:",
      (processed_dir / "restaurants_clustered.csv").exists())

In [ ]:
# Aggregate customer sentiment by restaurant
restaurant_sentiment = (
    reviews_clean
    .groupby("Restaurant")
    .agg(
        Total_Reviews=("Sentiment", "count"),
        Positive_Reviews=(
            "Sentiment",
            lambda x: (x == "Positive").sum()
        ),
        Negative_Reviews=(
            "Sentiment",
            lambda x: (x == "Negative").sum()
        ),
        Neutral_Reviews=(
            "Sentiment",
            lambda x: (x == "Neutral").sum()
        )
    )
    .reset_index()
)

# Calculate positive sentiment percentage
restaurant_sentiment["Positive_Percentage"] = (
    restaurant_sentiment["Positive_Reviews"]
    / restaurant_sentiment["Total_Reviews"]
    * 100
).round(2)

display(restaurant_sentiment.head())

In [ ]:
# Merge restaurant features and sentiment insights
restaurant_intelligence = restaurant_features.merge(
    restaurant_sentiment,
    left_on="Name",
    right_on="Restaurant",
    how="left"
)

# Check results
print("Combined dataset shape:", restaurant_intelligence.shape)

display(
    restaurant_intelligence[
        [
            "Name",
            "Cluster",
            "Cost",
            "Total_Reviews",
            "Positive_Percentage"
        ]
    ].head(10)
)

In [ ]:
print("Combined dataset shape:", restaurant_intelligence.shape)

print("\nMissing values:")
display(
    restaurant_intelligence[
        ["Cluster", "Cost", "Total_Reviews", "Positive_Percentage"]
    ].isnull().sum()
)

print("\nSample restaurant intelligence:")
display(
    restaurant_intelligence[
        [
            "Name",
            "Cost",
            "Cluster",
            "Total_Reviews",
            "Positive_Reviews",
            "Negative_Reviews",
            "Positive_Percentage"
        ]
    ].head(10)
)

In [ ]:
restaurant_intelligence.to_csv(
    processed_dir / "restaurant_intelligence.csv",
    index=False
)

print("Integrated restaurant intelligence dataset saved!")

In [ ]:
# Overall restaurant statistics
total_restaurants = restaurant_intelligence["Name"].nunique()

average_cost = restaurant_intelligence["Cost"].mean()

# Most common restaurant segment
largest_cluster = (
    restaurant_intelligence["Cluster"]
    .value_counts()
    .idxmax()
)

# Highest average cost segment
premium_cluster = (
    cluster_summary["Average_Cost"]
    .idxmax()
)

# Highest positive sentiment percentage
top_sentiment_restaurant = (
    restaurant_intelligence
    .dropna(subset=["Positive_Percentage"])
    .sort_values("Positive_Percentage", ascending=False)
    .iloc[0]
)

print("BUSINESS INSIGHTS")
print("-----------------")
print("Total Restaurants:", total_restaurants)
print("Average Restaurant Cost:", round(average_cost, 2))
print("Largest Restaurant Cluster:", largest_cluster)
print("Highest Average Cost Cluster:", premium_cluster)

print(
    "Highest Positive Sentiment Restaurant:",
    top_sentiment_restaurant["Name"]
)

print(
    "Positive Sentiment Percentage:",
    top_sentiment_restaurant["Positive_Percentage"]
)

In [ ]:
restaurant_intelligence.to_csv(
    processed_dir / "restaurant_intelligence.csv",
    index=False
)

print("Dashboard dataset ready!")